# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from api.getxapi_client import (
    fetch_replies,
    normalize_replies
)
from data.match_linking import (
    link_replies_to_matches,
    load_match_schedule,
)
from data.preprocess import (
    annotate_replies,
    preprocess_reply_buckets,
    preprocess_replies,
    preprocessing_summary
)
from models.sentiment_baseline import (
    add_sentiment,
    load_sentiment_model
)

In [2]:
response = fetch_replies(tweet_id="2069468618476200189")

response.keys()

dict_keys(['tweetId', 'reply_count', 'has_more', 'next_cursor', 'replies'])

In [3]:
rows = normalize_replies(
    response=response,
    match="Portugal vs Spain",
    event="goal",
    source_account="FIFAWorldCup",
    team="Portugal"
)

raw_df = pd.DataFrame(rows)

raw_df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source
0,2069547886975635584,2069468618476200189,https://x.com/NoCaptionMood/status/20695478869...,Tue Jun 23 22:27:21 +0000 2026,@FIFAWorldCup https://x.com/i/status/206952949...,NoCaptionMood,No Caption Mood,71,1,20265,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
1,2069547912825049173,2069468618476200189,https://x.com/grok/status/2069547912825049173,Tue Jun 23 22:27:27 +0000 2026,@NoCaptionMood @FIFAWorldCup Ask Grok is curre...,grok,Grok,111,1,11466,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
2,2069628055908392986,2069468618476200189,https://x.com/jopiew/status/2069628055908392986,Wed Jun 24 03:45:55 +0000 2026,@FIFAWorldCup Siuuuu...... https://t.co/32eMsS...,jopiew,Jops,0,0,27,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
3,2069468946906886206,2069468618476200189,https://x.com/Shameless_925/status/20694689469...,Tue Jun 23 17:13:40 +0000 2026,@FIFAWorldCup Stupid record. Messi has 18 WC g...,Shameless_925,Amanda❤️🌸,45,38,19389,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
4,2069488045565374770,2069468618476200189,https://x.com/TeeMacKuz/status/206948804556537...,Tue Jun 23 18:29:33 +0000 2026,@FIFAWorldCup When Ronaldo scores\nThe whole w...,TeeMacKuz,Oluwa Loba,553,8,14543,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi


In [4]:
raw_df.columns

Index(['tweet_id', 'parent_tweet_id', 'url', 'timestamp', 'text',
       'author_username', 'author_name', 'like_count', 'reply_count',
       'view_count', 'match', 'team', 'player', 'event', 'source_account',
       'source'],
      dtype='str')

In [5]:
raw_df.to_csv(
    "../data/raw/real_replies_sample.csv",
    index=False
)

In [6]:
raw_df.shape

(37, 16)

In [7]:
raw_df["text"].head(20)

0     @FIFAWorldCup https://x.com/i/status/206952949...
1     @NoCaptionMood @FIFAWorldCup Ask Grok is curre...
2     @FIFAWorldCup Siuuuu...... https://t.co/32eMsS...
3     @FIFAWorldCup Stupid record. Messi has 18 WC g...
4     @FIFAWorldCup When Ronaldo scores\nThe whole w...
5     To those who make others better. The first of ...
6     @FIFAWorldCup No one—I repeat, no one can ever...
7       @FIFAWorldCup cheers ⚽️ https://t.co/ijaapV5upe
8     @FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...
9     @FIFAWorldCup He has won how many World cup tr...
10    @FIFAWorldCup @pmcafrica THE GREATEST OF ALL T...
11         @FIFAWorldCup No,,😂🤣 https://t.co/YOBa5Qo8L4
12                @FIFAWorldCup https://t.co/F7vuPSwRPm
13                @FIFAWorldCup https://t.co/V2tTPIsyq2
14    90% of X users are engaging with World Cup con...
15    @FIFAWorldCup Ronaldo Celebrating his GOALS !!...
16    @FIFAWorldCup the GREATEST🫶🏻♾️ https://t.co/Q8...
17    @FIFAWorldCup MESSI and MBAPPE right now: 

In [8]:
reply_buckets = preprocess_reply_buckets(raw_df)

annotated_df = reply_buckets["annotated"]

schedule_df = load_match_schedule("../data/reference/matches_sample.csv")
annotated_df = link_replies_to_matches(
    annotated_df,
    schedule_df
)

analysis_df = annotated_df[annotated_df["filter_reason"] == "keep"].reset_index(drop=True)
review_df = analysis_df[analysis_df["needs_context_review"]].reset_index(drop=True)
removed_df = annotated_df[annotated_df["filter_reason"] != "keep"].reset_index(drop=True)

preprocessing_summary(annotated_df, analysis_df)

{'raw_rows': 37,
 'analysis_rows': 28,
 'removed_rows': 9,
 'filter_reasons': {'keep': 28, 'unusable_text': 8, 'spam': 1}}

In [9]:
pd.set_option("display.max_colwidth", None)

review_df[
    [
        "text",
        "clean_text",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score"
    ]
].head(20)

,text,clean_text,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score
0,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,3,[],1,True,1
1,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",3,"[achieve, achieved, man, no one, this man]",2,True,2
2,@FIFAWorldCup @pmcafrica THE GREATEST OF ALL TIME https://t.co/P6KOwPDIUK,THE GREATEST OF ALL TIME,3,[],1,True,1
3,@FIFAWorldCup the GREATEST🫶🏻♾️ https://t.co/Q8zz6ULbFB,the GREATEST🫶🏻♾️,3,[],1,True,1
4,@FIFAWorldCup Another record set! https://t.co/oCaTMNXXjZ,Another record set!,3,"[another record, record]",2,True,2
5,@FIFAWorldCup He knows who he is 🔥 https://t.co/Klny8X1YjO,He knows who he is 🔥,3,[he],2,True,2
6,@FIFAWorldCup HE’S FUCKING BACK BABYY😭 https://t.co/GAAOINIfiy,HE’S FUCKING BACK BABYY😭,3,"[back, he]",2,True,2
7,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,3,"[another record, record]",2,True,2


In [10]:
analysis_df[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "inferred_managers",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "linked_match_method",
        "entity_confidence",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score",
        "filter_reason"
    ]
].head(20)

,text,clean_text,matched_entities,inferred_teams,inferred_players,inferred_managers,linked_match_id,linked_match,linked_match_confidence,linked_match_method,entity_confidence,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score,filter_reason
0,@FIFAWorldCup Siuuuu...... https://t.co/32eMsSo3lZ,Siuuuu......,[siuuu],[Portugal],[Cristiano Ronaldo],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,1,3,[],0,False,2,keep
1,@FIFAWorldCup Stupid record. Messi has 18 WC goals https://t.co/CxPOqFJSWD,Stupid record. Messi has 18 WC goals,[messi],[Argentina],[Lionel Messi],[],2026_M045,Portugal vs Uzbekistan,medium,team_entity_and_timestamp,1,3,[record],2,False,4,keep
2,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,"[goat, messi, ronaldo]","[Argentina, Portugal]","[Cristiano Ronaldo, Lionel Messi]",[],2026_M045,Portugal vs Uzbekistan,medium,team_entity_and_timestamp,3,3,[],1,False,8,keep
3,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,[],[],[],[],NaN,NaN,none,no_nearby_match,0,3,[],1,True,1,keep
4,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",[],[],[],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,0,3,"[achieve, achieved, man, no one, this man]",2,True,2,keep
5,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,[messi],[Argentina],[Lionel Messi],[],2026_M045,Portugal vs Uzbekistan,medium,team_entity_and_timestamp,1,3,[],1,False,3,keep
6,@FIFAWorldCup He has won how many World cup trophies? @grok 😂😂 https://t.co/r3tM4hV8DA,He has won how many World cup trophies? 😂😂,"[cup, world cup]",[],[],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,2,3,[he],2,False,8,keep
7,@FIFAWorldCup @pmcafrica THE GREATEST OF ALL TIME https://t.co/P6KOwPDIUK,THE GREATEST OF ALL TIME,[],[],[],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,0,3,[],1,True,1,keep
8,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,"[cup, manager, world cup]",[],[],[],2026_M064,Uruguay vs Spain,high,team_entity_and_timestamp,3,3,[],1,False,10,keep
9,@FIFAWorldCup Ronaldo Celebrating his GOALS !! - ' I'M BACK' https://t.co/iUmyKXBYHE,Ronaldo Celebrating his GOALS !! - ' I'M BACK',[ronaldo],[Portugal],[Cristiano Ronaldo],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,1,3,"[back, his]",2,False,4,keep


In [11]:
classifier = load_sentiment_model()

analysis_df = add_sentiment(
    analysis_df,
    text_column="clean_text",
    classifier=classifier
)

analysis_df[
    [
        "text",
        "clean_text",
        "sentiment_label",
        "sentiment_score",
        "relevance_score",
        "needs_context_review"
    ]
].head(20)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,text,clean_text,sentiment_label,sentiment_score,relevance_score,needs_context_review
0,@FIFAWorldCup Siuuuu...... https://t.co/32eMsSo3lZ,Siuuuu......,NEGATIVE,0.951916,2,False
1,@FIFAWorldCup Stupid record. Messi has 18 WC goals https://t.co/CxPOqFJSWD,Stupid record. Messi has 18 WC goals,NEGATIVE,0.995983,4,False
2,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,POSITIVE,0.997308,8,False
3,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,To those who make others better. The first of many epic moments. Join MyLowe’s Rewards for free. Don’t miss what’s next.,POSITIVE,0.999742,1,True
4,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",NEGATIVE,0.997070,2,True
5,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,POSITIVE,0.995415,3,False
6,@FIFAWorldCup He has won how many World cup trophies? @grok 😂😂 https://t.co/r3tM4hV8DA,He has won how many World cup trophies? 😂😂,POSITIVE,0.992692,8,False
7,@FIFAWorldCup @pmcafrica THE GREATEST OF ALL TIME https://t.co/P6KOwPDIUK,THE GREATEST OF ALL TIME,POSITIVE,0.999836,1,True
8,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,POSITIVE,0.977773,10,False
9,@FIFAWorldCup Ronaldo Celebrating his GOALS !! - ' I'M BACK' https://t.co/iUmyKXBYHE,Ronaldo Celebrating his GOALS !! - ' I'M BACK',POSITIVE,0.999514,4,False


In [12]:
sample = analysis_df.sample(
    min(25, len(analysis_df)),
    random_state=42
)

sample[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "needs_context_review",
        "sentiment_label",
        "sentiment_score",
        "relevance_score"
    ]
]

,text,clean_text,matched_entities,inferred_teams,inferred_players,linked_match_id,linked_match,linked_match_confidence,needs_context_review,sentiment_label,sentiment_score,relevance_score
9,@FIFAWorldCup Ronaldo Celebrating his GOALS !! - ' I'M BACK' https://t.co/iUmyKXBYHE,Ronaldo Celebrating his GOALS !! - ' I'M BACK',[ronaldo],[Portugal],[Cristiano Ronaldo],2026_M045,Portugal vs Uzbekistan,high,False,POSITIVE,0.999514,4
25,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,[],[],[],2026_M045,Portugal vs Uzbekistan,high,True,POSITIVE,0.990748,2
8,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,90% of X users are engaging with World Cup conversations. Don't miss out: kick off your Engagements campaign in the new Ads Manager today.,"[cup, manager, world cup]",[],[],2026_M064,Uruguay vs Spain,high,False,POSITIVE,0.977773,10
21,"@FIFAWorldCup Well, goals against Uzbekistan doesn’t count https://t.co/HmY832oEZ6","Well, goals against Uzbekistan doesn’t count",[uzbekistan],[Uzbekistan],[],2026_M045,Portugal vs Uzbekistan,high,False,NEGATIVE,0.997355,3
0,@FIFAWorldCup Siuuuu...... https://t.co/32eMsSo3lZ,Siuuuu......,[siuuu],[Portugal],[Cristiano Ronaldo],2026_M045,Portugal vs Uzbekistan,high,False,NEGATIVE,0.951916,2
12,@FIFAWorldCup Relax… it’s just against Uzbekistan 🇺🇿 😂 https://t.co/mG8AG42zkb,Relax… it’s just against Uzbekistan 🇺🇿 😂,[uzbekistan],[Uzbekistan],[],2026_M045,Portugal vs Uzbekistan,high,False,POSITIVE,0.922164,3
17,@FIFAWorldCup CRISTIANO RONALDO IS THE NAME!!\nhttps://t.co/qfXt8tBCY5,CRISTIANO RONALDO IS THE NAME!!,"[cristiano, cristiano ronaldo, ronaldo]",[Portugal],[Cristiano Ronaldo],2026_M045,Portugal vs Uzbekistan,high,False,POSITIVE,0.999237,7
22,"@FIFAWorldCup Porto, Portugal 🇵🇹 https://t.co/HCg4iH82yt","Porto, Portugal 🇵🇹",[portugal],[Portugal],[],2026_M045,Portugal vs Uzbekistan,high,False,POSITIVE,0.962682,5
11,@FIFAWorldCup MESSI and MBAPPE right now: https://t.co/cZwaDLTMvt,MESSI and MBAPPE right now:,"[mbappe, messi]","[Argentina, France]","[Kylian Mbappé, Lionel Messi]",2026_M045,Portugal vs Uzbekistan,medium,False,POSITIVE,0.988174,5
13,@FIFAWorldCup Another record set! https://t.co/oCaTMNXXjZ,Another record set!,[],[],[],2026_M045,Portugal vs Uzbekistan,high,True,POSITIVE,0.998892,2


In [13]:
annotated_df.to_csv(
    "../data/processed/annotated_replies_sample.csv",
    index=False
)

analysis_df.to_csv(
    "../data/processed/analysis_replies_loose.csv",
    index=False
)

review_df.to_csv(
    "../data/processed/review_replies.csv",
    index=False
)

removed_df.to_csv(
    "../data/processed/removed_replies.csv",
    index=False
)

In [14]:
summary = {
    "raw_rows": len(raw_df),
    "annotated_rows": len(annotated_df),
    "analysis_rows": len(analysis_df),
    "review_rows": len(review_df),
    "removed_rows": len(removed_df),
    "linked_analysis_rows": analysis_df["linked_match_id"].notna().sum(),
    "ambiguous_analysis_rows": (analysis_df["linked_match_confidence"] == "ambiguous").sum(),
    "retention_rate": len(analysis_df) / len(raw_df)
}

summary

{'raw_rows': 37,
 'annotated_rows': 37,
 'analysis_rows': 28,
 'review_rows': 8,
 'removed_rows': 9,
 'linked_analysis_rows': np.int64(27),
 'ambiguous_analysis_rows': np.int64(0),
 'retention_rate': 0.7567567567567568}

### Observations

Real reply data contains media-only, text-poor, spam-like, low-relevance, context-dependent, and match-ambiguous replies. The current preprocessing is intentionally recall-oriented: it keeps more potentially relevant football discussion, including vague replies that rely on parent tweet context, while accepting that some junk can still slip into `analysis_df`.

The working pipeline is now:

```text
Raw API replies
-> normalized rows
-> clean text
-> football entity detection
-> parent-context relevance boost
-> inferred team/player/manager metadata
-> schedule-based match linking
-> data quality filtering
-> baseline sentiment labels and scores
```

`analysis_df` is the dataset sent to the baseline sentiment model. `review_df` is the subset of kept rows that rely mainly on parent context and should be manually inspected. `removed_df` keeps clear junk and unusable text for audit.

Match linking is conservative. It uses inferred teams, known parent context, and reply or parent timestamps to choose a scheduled match only when the evidence is strong enough. Rows can remain unlinked or ambiguous through `linked_match_confidence`, which is preferable to forcing bad match labels.

This loose filter is acceptable for the MVP because it prioritizes retaining real fan reactions over perfectly excluding every irrelevant reply. A future production version could add a second-stage relevance classifier or embedding similarity check to reduce false positives, and replace `matches_sample.csv` with a full official fixture schedule.

Filtering and enrichment are implemented in `src/data/preprocess.py`, `src/data/entities.py`, and `src/data/match_linking.py`, with reference data in `data/reference/football_entities.json` and `data/reference/matches_sample.csv`. Sentiment scoring is applied with the baseline DistilBERT model through `src/models/sentiment_baseline.py`.